In [8]:
"""
Autocorrelation analysis for LPBF photodiode scan-vector segments.

Computes the normalized autocorrelation for each segment in a layer,
extracts summary features, performs basic stationarity checks, and
produces diagnostic plots.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from typing import Optional
import pandas as pd

def normalized_autocorrelation(x: np.ndarray, max_lag: int) -> np.ndarray:
    """Compute normalized (biased) autocorrelation for lags 0..max_lag.

    Uses np.correlate in 'full' mode and normalizes by R(0) so that
    the output ranges in [-1, 1] with R[0] = 1.
    """
    x = x - np.mean(x)
    r0 = np.dot(x, x)
    if r0 == 0:
        return np.zeros(max_lag + 1)
    full = np.correlate(x, x, mode="full")
    # full is length 2N-1, centred at index N-1
    mid = len(x) - 1
    acf = full[mid : mid + max_lag + 1] / r0
    return acf


def halfside_stationarity_check(
    x: np.ndarray, max_lag: int, threshold: float = 0.25
) -> dict:
    """Compare autocorrelation of first vs second half of a segment.

    Returns the L2 norm of the difference between the two half-ACFs
    and a boolean flag if it exceeds *threshold*.
    """
    mid = len(x) // 2
    if mid < max_lag + 1:
        return {"acf_diff_norm": np.nan, "non_stationary": False}
    acf_first = normalized_autocorrelation(x[:mid], max_lag)
    acf_second = normalized_autocorrelation(x[mid:], max_lag)
    # Truncate to the shorter of the two (they should be equal or ±1)
    n = min(len(acf_first), len(acf_second))
    diff_norm = np.sqrt(np.mean((acf_first[:n] - acf_second[:n]) ** 2))
    return {"acf_diff_norm": float(diff_norm), "non_stationary": diff_norm > threshold}


def extract_acf_features(acf: np.ndarray, fs: float) -> dict:
    """Extract simple summary features from a normalized ACF.

    Parameters
    ----------
    acf : array, R[0] = 1
    fs  : sampling frequency in Hz

    Returns
    -------
    dict with:
        correlation_length_samples : lag at which |R| first drops below 1/e
        correlation_length_ms      : same, in milliseconds
        first_zero_crossing_lag    : first lag where R crosses zero (or NaN)
        first_secondary_peak_lag   : lag of first local max after the initial decay (or NaN)
        first_secondary_peak_value : R value at that peak
    """
    # --- correlation length (1/e threshold) ---
    below = np.where(np.abs(acf) < 1 / np.e)[0]
    corr_len = int(below[0]) if len(below) > 0 else len(acf)

    # --- first zero crossing ---
    sign_changes = np.where(np.diff(np.sign(acf)))[0]
    first_zero = int(sign_changes[0]) if len(sign_changes) > 0 else np.nan

    # --- first secondary peak (local max after first zero crossing) ---
    peak_lag = np.nan
    peak_val = np.nan
    if not np.isnan(first_zero):
        tail = acf[int(first_zero) :]
        local_maxima = []
        for i in range(1, len(tail) - 1):
            if tail[i] > tail[i - 1] and tail[i] > tail[i + 1]:
                local_maxima.append(i + int(first_zero))
        if local_maxima:
            peak_lag = int(local_maxima[0])
            peak_val = float(acf[peak_lag])

    return {
        "correlation_length_samples": corr_len,
        "correlation_length_ms": corr_len / fs * 1000,
        "first_zero_crossing_lag": first_zero,
        "first_secondary_peak_lag": peak_lag,
        "first_secondary_peak_value": peak_val,
    }


def analyze_layer_autocorrelation(
    segments: list[np.ndarray],
    fs: float = 20_000,
    max_lag_ms: float = 10.0,
    stationarity_threshold: float = 0.25,
    figsize: tuple = (14, 10),
    cmap_name: str = "viridis",
    title_prefix: str = "Layer",
) -> dict:
    """Full autocorrelation analysis and plotting for one layer.

    Parameters
    ----------
    segments : list of 1-D arrays
        Each array is the raw photodiode signal for one scan vector.
    fs : float
        Sampling rate in Hz (default 20 kHz).
    max_lag_ms : float
        Maximum lag to compute, in milliseconds.
    stationarity_threshold : float
        RMS ACF-difference threshold for the half-side stationarity check.
    figsize, cmap_name, title_prefix : plotting options.

    Returns
    -------
    dict with keys:
        acfs              : list of ACF arrays (one per segment)
        features          : list of feature dicts
        stationarity      : list of stationarity-check dicts
        lags_ms           : lag axis in ms
    """
    max_lag = int(max_lag_ms * fs / 1000)
    n_seg = len(segments)

    acfs = []
    features = []
    stationarity = []

    for seg in segments:
        # Skip degenerate segments
        if len(seg) < 2 * (max_lag + 1):
            effective_lag = max(len(seg) // 4, 1)
        else:
            effective_lag = max_lag

        acf = normalized_autocorrelation(seg, effective_lag)
        # Pad to uniform length for easy stacking
        if len(acf) < max_lag + 1:
            acf = np.pad(acf, (0, max_lag + 1 - len(acf)), constant_values=np.nan)

        acfs.append(acf)
        features.append(extract_acf_features(acf, fs))
        stationarity.append(
            halfside_stationarity_check(seg, min(effective_lag, len(seg) // 4), stationarity_threshold)
        )

    lags_ms = np.arange(max_lag + 1) / fs * 1000

    # ---- Plotting ----
    fig, axes = plt.subplots(2, 2, figsize=figsize, constrained_layout=True)
    fig.suptitle(f"{title_prefix} — Autocorrelation Diagnostics", fontsize=14, fontweight="bold")

    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=0, vmax=max(n_seg - 1, 1))

    # (a) Overlay of all ACFs, coloured by segment index
    ax = axes[0, 0]
    for i, acf in enumerate(acfs):
        ax.plot(lags_ms, acf, color=cmap(norm(i)), alpha=0.5, linewidth=0.7)
    ax.axhline(1 / np.e, ls="--", color="grey", lw=0.8, label="1/e threshold")
    ax.axhline(0, ls="-", color="black", lw=0.5)
    ax.set_xlabel("Lag (ms)")
    ax.set_ylabel("R(τ)")
    ax.set_title("(a) ACF per scan vector")
    ax.legend(fontsize=8)
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label="Segment index")

    # (b) Heatmap of ACFs (segment index vs lag)
    ax = axes[0, 1]
    acf_matrix = np.vstack(acfs)
    im = ax.imshow(
        acf_matrix,
        aspect="auto",
        extent=[lags_ms[0], lags_ms[-1], n_seg - 0.5, -0.5],
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
    )
    ax.set_xlabel("Lag (ms)")
    ax.set_ylabel("Segment index")
    ax.set_title("(b) ACF heatmap across layer")
    fig.colorbar(im, ax=ax, label="R(τ)")

    # (c) Correlation length evolution
    ax = axes[1, 0]
    corr_lengths = [f["correlation_length_ms"] for f in features]
    colors_c = [
        "red" if s["non_stationary"] else cmap(norm(i))
        for i, s in enumerate(stationarity)
    ]
    ax.bar(range(n_seg), corr_lengths, color=colors_c, width=1.0, edgecolor="none")
    ax.set_xlabel("Segment index")
    ax.set_ylabel("Correlation length (ms)")
    ax.set_title("(c) Correlation length (red = non-stationary flag)")

    # (d) Stationarity check: ACF difference norm
    ax = axes[1, 1]
    diff_norms = [s["acf_diff_norm"] for s in stationarity]
    flags = [s["non_stationary"] for s in stationarity]
    bar_colors = ["red" if f else "steelblue" for f in flags]
    ax.bar(range(n_seg), diff_norms, color=bar_colors, width=1.0, edgecolor="none")
    ax.axhline(stationarity_threshold, ls="--", color="grey", lw=1, label=f"threshold = {stationarity_threshold}")
    ax.set_xlabel("Segment index")
    ax.set_ylabel("RMS ACF difference (1st vs 2nd half)")
    ax.set_title("(d) Half-segment stationarity check")
    ax.legend(fontsize=8)

    plt.savefig("autocorrelation_diagnostics.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return {
        "acfs": acfs,
        "features": features,
        "stationarity": stationarity,
        "lags_ms": lags_ms,
    }


# ---- Demo with synthetic data ----
if __name__ == "__main__":
    fs = 20_000
    df = pd.read_csv("csvs3/layer200.csv")
    segments = []
    for scan_num in range(int(df["scan_number"].max()) + 1):
        segments.append(df[df["scan_number"] == scan_num]["signal"].values)

    def make_segment(n_samples, freq_hz, noise_std, drift_rate=0.0):
        """Synthetic photodiode-like segment: sine + noise + optional drift."""
        t = np.arange(n_samples) / fs
        signal = np.sin(2 * np.pi * freq_hz * t)
        signal += drift_rate * t  # linear drift
    results = analyze_layer_autocorrelation(
        segments, fs=fs, max_lag_ms=5.0, title_prefix="Synthetic Layer"
    )

    # Print a summary table
    print(f"{'Seg':>4s}  {'CorrLen(ms)':>11s}  {'1st Zero':>8s}  {'2nd Peak':>8s}  {'Stationarity':>12s}")
    print("-" * 55)
    for i, (feat, stat) in enumerate(zip(results["features"], results["stationarity"])):
        zc = f"{feat['first_zero_crossing_lag']}" if not np.isnan(feat["first_zero_crossing_lag"]) else "—"
        pk = f"{feat['first_secondary_peak_lag']}" if not np.isnan(feat["first_secondary_peak_lag"]) else "—"
        flag = "NON-STAT" if stat["non_stationary"] else "ok"
        print(f"{i:4d}  {feat['correlation_length_ms']:11.3f}  {zc:>8s}  {pk:>8s}  {flag:>12s}")

 Seg  CorrLen(ms)  1st Zero  2nd Peak  Stationarity
-------------------------------------------------------
   0        0.950        37         —            ok
   1        0.650        33         —            ok
   2        0.450        19        37            ok
   3        0.250        15        37            ok
   4        0.200         6        17            ok
   5        0.350        12        22      NON-STAT
   6        0.400        15        21      NON-STAT
   7        0.850        30         —            ok
   8        0.400         9         —            ok
   9        0.250         9        11            ok
  10        0.450        24         —            ok
  11        0.350        32        35            ok
  12        0.400        32         —      NON-STAT
  13        0.300        16        20      NON-STAT
  14        0.400        30        50      NON-STAT
  15        0.600        21        26            ok
  16        0.850        43        55            ok
  17    